# 05 · IA generativa: asistente de consultas SQL (Módulo 5)
**Sistema de apoyo para detección de tumores cerebrales en resonancia magnética**

**Qué problema real resuelve.** El Administrador del hospital necesita respuestas rápidas sobre el uso del sistema:
cuántos estudios con posible tumor hubo esta semana, qué doctor está en desacuerdo con el modelo, qué modelo tiene
más falsos negativos, cuánto tarda cada predicción... Esa información vive en el modelo estrella (SQL Server / Fabric),
pero escribir SQL no es parte de su trabajo. El asistente traduce una pregunta en español a **una consulta de solo lectura**,
la ejecuta con controles de seguridad y redacta la respuesta en lenguaje sencillo.

**Flujo (dos llamadas a Claude):**
1. *Generar la consulta:* Claude recibe la pregunta y el esquema de una **lista blanca** de vistas (sin hashes ni identificadores
   de pacientes) y devuelve, con **salida estructurada** (JSON Schema), `puede_responder`, `sql`, `explicacion` y `motivo`.
2. *Validar* con `sqlglot`, sin excepciones: una sola sentencia `SELECT`, sin DML/DDL en ningún nodo, solo vistas de la lista
   blanca (también en subconsultas y CTE), sin funciones peligrosas ni comentarios, con **límite de filas forzado**.
3. *Ejecutar* en una conexión de **solo lectura** con tiempo máximo.
4. *Redactar:* Claude recibe la pregunta, la consulta y como máximo 50 filas, y escribe la respuesta.

Todo queda en la bitácora (`CONSULTA_IA` o `CONSULTA_IA_BLOQUEADA`). El código está en `app/backend/asistente.py`; este notebook
**importa los prompts de ese módulo**, así que lo que se muestra aquí es exactamente lo que usa la aplicación.
La clave de la API vive solo en `.env` y nunca se muestra.

In [1]:
import os
import sys
from pathlib import Path

RAIZ = Path.cwd().resolve()
while not (RAIZ / "app" / "backend").is_dir():
    if RAIZ.parent == RAIZ:
        raise RuntimeError("No se encontró la raíz del proyecto.")
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

import pandas as pd
from app.backend import asistente, config, db

pd.set_option("display.max_colwidth", 120)
db.crear_esquema()   # crea las vistas de la lista blanca si faltan (no borra nada)
print("Modelo:", config.ANTHROPIC_MODELO)
print("Base:", config.DB_MODO, "| dialecto SQL:", asistente.dialecto())
print("¿Hay clave configurada?:", config.asistente_configurado())   # solo sí/no; la clave nunca se imprime
print("Límite de filas:", config.ASISTENTE_LIMITE_FILAS, "| tiempo máximo de la consulta:", config.ASISTENTE_SEGUNDOS_SQL, "s")
print("Parámetros: max_tokens =", asistente.MAX_TOKENS, "| esfuerzo =", asistente.ESFUERZO,
      "| respaldo del servidor =", asistente.BETA_RESPALDO if config.ASISTENTE_RESPALDO else "desactivado")

Modelo: claude-opus-5-5
Base: local | dialecto SQL: sqlite
¿Hay clave configurada?: True
Límite de filas: 200 | tiempo máximo de la consulta: 10 s
Parámetros: max_tokens = 4000 | esfuerzo = medium | respaldo del servidor = server-side-fallback-2026-07-01


## 1. Prompts exactos
### 1.1 Prompt del sistema para generar la consulta
Variables que se sustituyen: `{motor}`, `{dialecto}`, `{esquema}` (la lista blanca) y `{limite}`.

In [2]:
print(asistente.prompt_sistema_sql())

Eres el asistente de consultas de un tablero hospitalario académico que apoya la detección de tumores cerebrales en resonancias. Ayudas al Administrador a obtener datos con UNA consulta SQL de solo lectura.

Motor: SQLite (dialecto sqlite).

Reglas obligatorias:
1. Escribe UNA sola sentencia SELECT (se permite WITH ... SELECT). Nunca INSERT, UPDATE, DELETE, DROP, ALTER, CREATE, PRAGMA, ATTACH, EXEC ni nada que cambie datos.
2. Usa solo estas vistas y columnas. No existen otras tablas para ti:
- vw_predicciones: Una fila por predicción. origen='uso' son estudios reales subidos por doctores; origen='prueba' son los resultados de los 5 modelos sobre el conjunto de prueba.
    * prediccion_id: número de la predicción
    * origen: 'uso' o 'prueba'
    * creado_en: fecha y hora en que se registró
    * fecha: fecha del estudio
    * anio: año de la fecha del estudio
    * mes: mes (1-12)
    * nombre_mes: nombre del mes en español
    * semana: semana ISO del año
    * dia_semana: día de la

### 1.2 Mensaje del usuario para generar la consulta
Variables: `{hoy}`, `{anio}`, `{semana}` y `{pregunta}` (con los identificadores de paciente ocultos).

In [3]:
print(asistente.PLANTILLA_USUARIO_SQL)
print("--- ejemplo ---")
print(asistente.prompt_usuario_sql("¿Cuántos estudios con posible tumor hubo esta semana por doctor?"))

Fecha de hoy: {hoy} (año {anio}, semana ISO {semana}).

<pregunta>
{pregunta}
</pregunta>
--- ejemplo ---
Fecha de hoy: 2026-11-21 (año 2026, semana ISO 47).

<pregunta>
¿Cuántos estudios con posible tumor hubo esta semana por doctor?
</pregunta>


### 1.3 Esquema de la salida estructurada (JSON Schema)

In [4]:
import json
print(json.dumps(asistente.ESQUEMA_SALIDA_SQL, indent=2, ensure_ascii=False))

{
  "type": "object",
  "properties": {
    "puede_responder": {
      "type": "boolean"
    },
    "sql": {
      "type": "string"
    },
    "explicacion": {
      "type": "string"
    },
    "motivo": {
      "type": "string"
    }
  },
  "required": [
    "puede_responder",
    "sql",
    "explicacion",
    "motivo"
  ],
  "additionalProperties": false
}


### 1.4 Prompt del sistema y mensaje para redactar la respuesta
Variables: `{pregunta}`, `{sql}`, `{n_filas}`, `{n_mostradas}`, `{columnas}` y `{filas}` (máximo 50, en JSON).

In [5]:
print(asistente.PROMPT_SISTEMA_RESPUESTA)
print("--- plantilla ---")
print(asistente.PLANTILLA_USUARIO_RESPUESTA)

Eres el asistente de consultas de un tablero hospitalario académico de apoyo a la detección de tumores cerebrales. Recibes la pregunta del Administrador, la consulta SQL que se ejecutó y sus resultados. Redacta la respuesta en español de México, formal, sencillo y claro, en 1 a 4 oraciones o una lista corta.

Reglas:
- Usa solo los números que aparecen en los resultados; no inventes ni extrapoles. Si no hay filas, dilo.
- Si los resultados vienen recortados, indícalo.
- No des diagnósticos médicos ni recomendaciones clínicas; es información de apoyo sobre el uso del sistema.
- No menciones detalles técnicos de SQL salvo que ayuden a entender el resultado.
--- plantilla ---
<pregunta>
{pregunta}
</pregunta>

<consulta_sql>
{sql}
</consulta_sql>

<resultados filas_totales="{n_filas}" filas_mostradas="{n_mostradas}">
columnas: {columnas}
{filas}
</resultados>


## 2. Lista blanca
Solo vistas de lectura. **No** están `usuarios` (hash de contraseñas), `sesiones`, `dim_paciente` (identificador del paciente)
ni las tablas base. `vw_predicciones` trae edad, grupo de edad y sexo ficticios, pero no el identificador.

In [6]:
filas = [(vista, desc, len(cols)) for vista, (desc, cols) in asistente.ESQUEMA.items()]
display(pd.DataFrame(filas, columns=["vista", "descripción", "columnas"]))

,vista,descripción,columnas
0,vw_predicciones,Una fila por predicción. origen='uso' son estudios reales subidos por doctores; origen='prueba' son los resultados d...,30
1,vw_estudios_semana,Estudios de uso por semana ISO (de la fecha del estudio) y doctor.,8
2,vw_metricas_modelo,"Matriz de confusión por versión de modelo y origen, solo con casos confirmados.",12
3,vw_modelos,Catálogo de modelos.,14
4,vw_entrenamientos,Entrenamientos y reentrenamientos de la red.,10
5,vw_metricas_pliegue,Métricas por pliegue de la validación cruzada.,13
6,vw_historial_epocas,Historial por época de cada entrenamiento.,8
7,vw_bitacora,Bitácora de acciones del sistema.,7
8,dim_fecha,Calendario.,7
9,dim_diagnostico,Catálogo de diagnósticos.,4


## 3. El validador
Casos peligrosos (todos deben quedar bloqueados) y casos válidos (se les fuerza el límite de filas).

In [7]:
casos = {
    "Borrar tabla": "DROP TABLE usuarios",
    "Cambiar rol": "UPDATE usuarios SET rol='administrador'",
    "Leer hashes": "SELECT hash_password FROM usuarios",
    "Identificadores": "SELECT * FROM dim_paciente",
    "Subconsulta fuera de lista": "SELECT nombre FROM vw_modelos WHERE modelo_id IN (SELECT usuario_id FROM usuarios)",
    "CTE fuera de lista": "WITH u AS (SELECT * FROM usuarios) SELECT * FROM u",
    "Dos sentencias": "SELECT * FROM vw_modelos; DROP TABLE usuarios",
    "Comentario que esconde otra": "SELECT * FROM vw_modelos /* ; DROP TABLE usuarios */",
    "PRAGMA": "PRAGMA table_info(usuarios)",
    "ATTACH": "ATTACH DATABASE 'otra.db' AS otra",
    "Función peligrosa": "SELECT load_extension('x')",
    "SELECT ... INTO": "SELECT * INTO copia FROM vw_modelos",
    "Válida (agregada)": "SELECT modelo, COUNT(*) AS n FROM vw_predicciones WHERE origen = 'prueba' GROUP BY modelo",
    "Válida (pide 100000 filas)": "SELECT * FROM vw_bitacora LIMIT 100000",
}
resultado = []
for nombre, sql in casos.items():
    try:
        final = asistente.validar_sql(sql)
        resultado.append((nombre, "aceptada", final))
    except asistente.Bloqueada as e:
        resultado.append((nombre, "BLOQUEADA", str(e)))
tabla = pd.DataFrame(resultado, columns=["caso", "resultado", "SQL final o motivo"])
display(tabla)
assert (tabla["resultado"] == "BLOQUEADA").sum() == len(casos) - 2

,caso,resultado,SQL final o motivo
0,Borrar tabla,BLOQUEADA,La consulta contiene una instrucción no permitida: DROP.
1,Cambiar rol,BLOQUEADA,La consulta contiene una instrucción no permitida: UPDATE.
2,Leer hashes,BLOQUEADA,La tabla o vista 'usuarios' no está en la lista blanca.
3,Identificadores,BLOQUEADA,La tabla o vista 'dim_paciente' no está en la lista blanca.
4,Subconsulta fuera de lista,BLOQUEADA,La tabla o vista 'usuarios' no está en la lista blanca.
5,CTE fuera de lista,BLOQUEADA,La tabla o vista 'usuarios' no está en la lista blanca.
6,Dos sentencias,BLOQUEADA,Solo se permite una sentencia.
7,Comentario que esconde otra,BLOQUEADA,La consulta no puede llevar comentarios.
8,PRAGMA,BLOQUEADA,La consulta contiene una instrucción no permitida: PRAGMA.
9,ATTACH,BLOQUEADA,La consulta contiene una instrucción no permitida: ATTACH.


### 3.1 Ejecución de una consulta válida en la base local (solo lectura)

In [8]:
sql = asistente.validar_sql(
    "SELECT modelo, modelo_version, origen, casos, vp, vn, fp, fn, "
    "ROUND(1.0 * vp / NULLIF(vp + fn, 0), 4) AS sensibilidad, ROUND(1.0 * vn / NULLIF(vn + fp, 0), 4) AS especificidad "
    "FROM vw_metricas_modelo ORDER BY origen, modelo")
print(sql)
columnas, filas = asistente.ejecutar(sql)
display(pd.DataFrame(filas, columns=columnas))

SELECT modelo, modelo_version, origen, casos, vp, vn, fp, fn, ROUND(1.0 * vp / NULLIF(vp + fn, 0), 4) AS sensibilidad, ROUND(1.0 * vn / NULLIF(vn + fp, 0), 4) AS especificidad FROM vw_metricas_modelo ORDER BY origen, modelo LIMIT 200


,modelo,modelo_version,origen,casos,vp,vn,fp,fn,sensibilidad,especificidad
0,Logística (MobileNetV2),logistica_mobilenet,prueba,720,633,63,10,14,0.9784,0.8630
1,Logística (rasgos básicos),logistica_basica,prueba,720,632,22,51,15,0.9768,0.3014
2,Random Forest (MobileNetV2),random_forest,prueba,720,636,58,15,11,0.9830,0.7945
3,Red Keras (MobileNetV2),v001,prueba,720,626,65,8,21,0.9675,0.8904
4,SVM RBF (MobileNetV2),svm_rbf,prueba,720,633,63,10,14,0.9784,0.8630
5,Red Keras (MobileNetV2),v001,uso,3,1,1,1,0,1.0000,0.5000


## 4. Llamada real a la API de Claude
Si hay clave en `.env`, se hace **una** pregunta real (dos llamadas: generar y redactar) y se registra en la bitácora.
Si no hay clave, se indica y el notebook sigue sin error.

In [9]:
if config.asistente_configurado():
    r = asistente.responder("¿Cuántos estudios con posible tumor hubo esta semana por doctor?",
                            {"usuario_id": None, "usuario": "notebook_05", "rol": "administrador"}, "notebook",
                            asistente.obtener_cliente())
    print("Estado:", r["estado"])
    print("SQL:", r["sql"])
    if r["columnas"]:
        display(pd.DataFrame(r["filas"], columns=r["columnas"]))
    print("Respuesta:", r["respuesta"])
else:
    print("No hay ANTHROPIC_API_KEY en .env: se omite la llamada real (el resto del notebook no la necesita).")

Estado: ok
SQL: SELECT doctor, SUM(con_posible_tumor) AS estudios_con_posible_tumor, SUM(estudios) AS estudios_totales FROM vw_estudios_semana WHERE anio = 2026 AND semana = 47 GROUP BY doctor ORDER BY estudios_con_posible_tumor DESC LIMIT 200


,doctor,estudios_con_posible_tumor,estudios_totales
0,doctor1,2,3


Respuesta: En la semana 47 de 2026, solo un doctor registró estudios en el sistema:

- **doctor1:** 2 estudios con posible tumor de un total de 3 estudios realizados.

Ningún otro doctor tuvo estudios registrados en esa semana.


## 5. Reglas de seguridad (resumen)
- Solo `SELECT` sobre la lista blanca, una sentencia, sin comentarios, sin DML/DDL, sin `PRAGMA`/`ATTACH`/`EXEC`, sin funciones
  peligrosas; límite de filas forzado (`LIMIT` en SQLite, `TOP` en T-SQL).
- Conexión de solo lectura (SQLite `mode=ro` + `query_only`; en Fabric `ApplicationIntent=ReadOnly` y un usuario con solo lectura).
- No se envían a la API datos que identifiquen pacientes: la lista blanca no los tiene, los identificadores escritos en la
  pregunta se ocultan y se quitan columnas sensibles antes de enviar filas.
- Toda consulta y todo rechazo queda en la bitácora. La clave nunca se muestra, ni se registra, ni se devuelve.
- Es información de apoyo sobre el uso del sistema; no es un diagnóstico médico.